In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 4 — Pipeline Orchestrator
# MAGIC
# MAGIC Runs Bronze, Silver, and Gold in sequence.
# MAGIC Supports two modes:
# MAGIC - `full_refresh`
# MAGIC - `incremental`
# MAGIC
# MAGIC Example Databricks Job parameter:
# MAGIC ```
# MAGIC {"mode": "incremental"}
# MAGIC ```

# COMMAND ----------
# MAGIC %md ## 0. Read run mode

# COMMAND ----------

try:
    MODE = dbutils.widgets.get("mode")
except Exception:
    MODE = "full_refresh"

print(f"Run mode: {MODE}")

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime

CATALOG = "workspace"
SCHEMA  = "medallion"

current_user = spark.sql("SELECT current_user()").collect()[0][0]

# Keep RAW_PATH unchanged
# RAW_PATH = f"/Workspace/Users/{current_user}/raw_data"
RAW_PATH =f"/Volumes/workspace/default/course_data/Promotion_raw_data"
RUN_ID     = datetime.now().strftime("%Y%m%d_%H%M%S")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Raw path: {RAW_PATH}")


# COMMAND ----------
# MAGIC %md ## 1. Metadata tables

# COMMAND ----------

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_watermarks (
    table_name      STRING,
    last_order_date DATE,
    rows_processed  BIGINT,
    run_status      STRING,
    run_mode        STRING,
    updated_at      TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_run_log (
    run_id       STRING,
    run_mode     STRING,
    stage        STRING,
    rows_in      BIGINT,
    rows_out     BIGINT,
    status       STRING,
    error_msg    STRING,
    started_at   TIMESTAMP,
    finished_at  TIMESTAMP
)
USING DELTA
""")

def get_watermark(table_name: str) -> str:
    rows = spark.sql(f"""
        SELECT CAST(last_order_date AS STRING) AS last_order_date
        FROM {CATALOG}.{SCHEMA}.pipeline_watermarks
        WHERE table_name = '{table_name}'
          AND run_status = 'SUCCESS'
        ORDER BY updated_at DESC
        LIMIT 1
    """).collect()
    wm = rows[0][0] if rows else "1900-01-01"
    print(f"  Watermark [{table_name}]: {wm}")
    return wm

def set_watermark(table_name: str, last_date: str, rows: int, status="SUCCESS"):
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_watermarks
        VALUES (
            '{table_name}',
            DATE('{last_date}'),
            {rows},
            '{status}',
            '{MODE}',
            CURRENT_TIMESTAMP()
        )
    """)
    print(f"  Watermark updated [{table_name}] → {last_date} ({rows:,} rows, {status})")

def log(stage, rows_in=0, rows_out=0, status="SUCCESS", error=""):
    safe_error = (error or "")[:200].replace("'", "")
    spark.sql(f"""
        INSERT INTO {CATALOG}.{SCHEMA}.pipeline_run_log
        VALUES (
            '{RUN_ID}',
            '{MODE}',
            '{stage}',
            {rows_in},
            {rows_out},
            '{status}',
            '{safe_error}',
            CURRENT_TIMESTAMP(),
            CURRENT_TIMESTAMP()
        )
    """)


# COMMAND ----------
# MAGIC %md ## 2. Shared helpers

# COMMAND ----------

def read_raw_csv(filename):
    return (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("escape", '"')
        .csv(f"{RAW_PATH}/{filename}")
    )

def clean_column_name(c: str) -> str:
    return (
        c.strip()
         .replace(" ", "_")
         .replace(",", "_")
         .replace(";", "_")
         .replace("{", "_")
         .replace("}", "_")
         .replace("(", "_")
         .replace(")", "_")
         .replace("\n", "_")
         .replace("\t", "_")
         .replace("=", "_")
         .replace("-", "_")
    )

def standardize_columns(df):
    return df.select([F.col(c).alias(clean_column_name(c)) for c in df.columns])

def rename_if_exists(df, old_name, new_name):
    return df.withColumnRenamed(old_name, new_name) if old_name in df.columns else df

def drop_if_exists(df, cols):
    existing = [c for c in cols if c in df.columns]
    return df.drop(*existing) if existing else df

def trim_all_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.trim(F.col(field.name)))
    return df

def parse_date(col_name):
    c = f"`{col_name}`"
    return F.coalesce(
        F.to_date(F.expr(f"try_to_timestamp({c}, 'yyyy-MM-dd')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'MM/dd/yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'dd-MM-yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'yyyyMMdd')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'MMM dd, yyyy')")),
        F.to_date(F.expr(f"try_to_timestamp({c}, 'yyyy-MM-dd HH:mm:ss')"))
    )

def clean_price(col_name):
    return F.regexp_replace(F.col(col_name), r"[^\d\.\-]", "").cast(DoubleType())

def write_table(df, full_table_name, mode="overwrite"):
    (
        df.write
        .format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

def table_exists(full_table_name: str) -> bool:
    try:
        spark.table(full_table_name)
        return True
    except Exception:
        return False


# COMMAND ----------
# MAGIC %md ## 3. Bronze stage

# COMMAND ----------

def run_bronze():
    print("\n▓▓▓  STAGE 1/3 — BRONZE  ▓▓▓")
    start = datetime.now()

    def ingest(filename, table_name, mode="overwrite"):
        full_table_name = f"{CATALOG}.{SCHEMA}.bronze_{table_name}"

        df = read_raw_csv(filename)
        df = standardize_columns(df)

        df = (
            df.withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path"))
        )

        write_table(df, full_table_name, mode=mode)

        n = spark.table(full_table_name).count()
        print(f"  {full_table_name:45s} {n:>8,} rows")
        return n

    # Dimensions always full refresh
    ingest("RAW_Customer.csv", "customer", mode="overwrite")
    ingest("RAW_Product.csv", "product", mode="overwrite")
    ingest("RAW_Reseller.csv", "reseller", mode="overwrite")
    ingest("RAW_SalesTerritory.csv", "sales_territory", mode="overwrite")
    ingest("RAW_Date.csv", "date", mode="overwrite")
    ingest("RAW_SalesOrder.csv", "sales_order", mode="overwrite")

    # Fact table
    full_table_name = f"{CATALOG}.{SCHEMA}.bronze_sales"

    df_sales_raw = read_raw_csv("RAW_Sales.csv")
    df_sales_raw = standardize_columns(df_sales_raw)

    # Expect Order_Date after standardization
    if "Order_Date" in df_sales_raw.columns:
        df_sales_raw = df_sales_raw.withColumn("_parsed_date", parse_date("Order_Date"))
    else:
        raise ValueError("RAW_Sales.csv does not contain Order_Date column after standardization")

    wm = get_watermark("bronze_sales") if MODE == "incremental" else "1900-01-01"

    df_new = df_sales_raw.filter(F.col("_parsed_date") > F.lit(wm)).drop("_parsed_date")
    new_rows = df_new.count()

    if new_rows == 0:
        print("  bronze_sales                                  0 new rows — skipping")
        log("bronze", rows_out=0)
        return

    df_new = (
        df_new.withColumn("_ingest_time", F.current_timestamp())
              .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    write_mode = "overwrite" if MODE == "full_refresh" else "append"
    write_table(df_new, full_table_name, mode=write_mode)

    max_date = df_new.withColumn("_od", parse_date("Order_Date")).agg(F.max("_od")).collect()[0][0]
    set_watermark("bronze_sales", str(max_date), new_rows)

    print(f"  {full_table_name:45s} {new_rows:>8,} rows  (up to {max_date})")

    elapsed = (datetime.now() - start).seconds
    log("bronze", rows_out=new_rows)
    print(f"  ✔ Bronze done in {elapsed}s")


# COMMAND ----------
# MAGIC %md ## 4. Silver stage

# COMMAND ----------

def run_silver():
    print("\n▓▓▓  STAGE 2/3 — SILVER  ▓▓▓")
    start = datetime.now()

    BRONZE_CUSTOMER       = f"{CATALOG}.{SCHEMA}.bronze_customer"
    BRONZE_PRODUCT        = f"{CATALOG}.{SCHEMA}.bronze_product"
    BRONZE_RESELLER       = f"{CATALOG}.{SCHEMA}.bronze_reseller"
    BRONZE_TERRITORY      = f"{CATALOG}.{SCHEMA}.bronze_sales_territory"
    BRONZE_DATE           = f"{CATALOG}.{SCHEMA}.bronze_date"
    BRONZE_SALES_ORDER    = f"{CATALOG}.{SCHEMA}.bronze_sales_order"
    BRONZE_SALES          = f"{CATALOG}.{SCHEMA}.bronze_sales"

    SILVER_DIM_CUSTOMER   = f"{CATALOG}.{SCHEMA}.silver_dim_customer"
    SILVER_DIM_PRODUCT    = f"{CATALOG}.{SCHEMA}.silver_dim_product"
    SILVER_DIM_RESELLER   = f"{CATALOG}.{SCHEMA}.silver_dim_reseller"
    SILVER_DIM_TERRITORY  = f"{CATALOG}.{SCHEMA}.silver_dim_territory"
    SILVER_DIM_DATE       = f"{CATALOG}.{SCHEMA}.silver_dim_date"
    SILVER_DIM_SALESORDER = f"{CATALOG}.{SCHEMA}.silver_dim_salesorder"
    SILVER_FACT_SALES     = f"{CATALOG}.{SCHEMA}.silver_fact_sales"

    # dim_customer
    df = spark.table(BRONZE_CUSTOMER)
    df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date", "_source"])

    df = rename_if_exists(df, "Customer_ID", "CustomerNaturalKey")
    df = rename_if_exists(df, "Customer", "CustomerName")
    df = rename_if_exists(df, "State_Province", "StateProvince")
    df = rename_if_exists(df, "Country_Region", "CountryRegion")
    df = rename_if_exists(df, "Postal_Code", "PostalCode")

    df = trim_all_string_columns(df)

    if "StateProvince" in df.columns:
        df = df.withColumn("StateProvince", F.initcap(F.col("StateProvince")))
    if "CountryRegion" in df.columns:
        df = df.withColumn("CountryRegion", F.initcap(F.col("CountryRegion")))
    if "City" in df.columns:
        df = df.withColumn("City", F.when(F.col("City").isNull() | (F.col("City") == ""), "Unknown").otherwise(F.col("City")))
    if "PostalCode" in df.columns:
        df = df.withColumn("PostalCode", F.when(F.col("PostalCode").isNull() | (F.col("PostalCode") == ""), "00000").otherwise(F.col("PostalCode")))
    if "CustomerName" in df.columns:
        df = df.withColumn("CustomerName", F.when(F.col("CustomerName").isNull() | (F.col("CustomerName") == ""), "Unknown").otherwise(F.col("CustomerName")))

    df = df.dropDuplicates(["CustomerNaturalKey"])
    df = df.withColumn("CustomerKey", F.row_number().over(Window.orderBy("CustomerNaturalKey")))
    df = df.select("CustomerKey", "CustomerNaturalKey", "CustomerName", "City", "StateProvince", "CountryRegion", "PostalCode")
    write_table(df, SILVER_DIM_CUSTOMER)
    print(f"  {SILVER_DIM_CUSTOMER:45s} {spark.table(SILVER_DIM_CUSTOMER).count():>8,} rows")

    # dim_product
    df = spark.table(BRONZE_PRODUCT)
    df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])
    df = rename_if_exists(df, "Standard_Cost", "StandardCost")
    df = rename_if_exists(df, "List_Price", "ListPrice")

    df = trim_all_string_columns(df)
    df = df.withColumn("ListPrice", clean_price("ListPrice"))
    df = df.withColumn("StandardCost", clean_price("StandardCost"))
    df = df.filter(F.col("ListPrice") > 0)

    if "Color" in df.columns:
        df = df.withColumn("Color", F.when(F.col("Color").isNull() | (F.col("Color") == ""), "No Color").otherwise(F.initcap(F.col("Color"))))
    if "Category" in df.columns:
        df = df.withColumn("Category", F.when(F.col("Category").isNull() | (F.col("Category") == ""), "Unknown").otherwise(F.initcap(F.col("Category"))))
    if "Subcategory" in df.columns:
        df = df.withColumn("Subcategory", F.initcap(F.col("Subcategory")))

    df = df.withColumn("StandardCost", F.when(F.col("StandardCost").isNull(), 0.0).otherwise(F.col("StandardCost")))
    df = df.dropDuplicates(["SKU"])
    df = df.withColumn("ProductKey", F.row_number().over(Window.orderBy("SKU")))
    df = df.select("ProductKey", "SKU", "Product", "Model", "Category", "Subcategory", "Color", "ListPrice", "StandardCost")
    write_table(df, SILVER_DIM_PRODUCT)
    print(f"  {SILVER_DIM_PRODUCT:45s} {spark.table(SILVER_DIM_PRODUCT).count():>8,} rows")

    # dim_reseller
    df = spark.table(BRONZE_RESELLER)
    df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])

    df = rename_if_exists(df, "Reseller_ID", "ResellerNaturalKey")
    df = rename_if_exists(df, "Business_Type", "BusinessType")
    df = rename_if_exists(df, "State_Province", "StateProvince")
    df = rename_if_exists(df, "Country_Region", "CountryRegion")
    df = rename_if_exists(df, "Postal_Code", "PostalCode")

    df = trim_all_string_columns(df)

    if "BusinessType" in df.columns:
        df = df.withColumn("BusinessType", F.when(F.col("BusinessType").isNull() | (F.col("BusinessType") == ""), "Unknown").otherwise(F.initcap(F.col("BusinessType"))))
    if "City" in df.columns:
        df = df.withColumn("City", F.when(F.col("City").isNull() | (F.col("City") == ""), "Unknown").otherwise(F.col("City")))
    if "PostalCode" in df.columns:
        df = df.withColumn("PostalCode", F.when(F.col("PostalCode").isNull() | (F.col("PostalCode") == ""), "00000").otherwise(F.col("PostalCode")))
    if "CountryRegion" in df.columns:
        df = df.withColumn("CountryRegion", F.initcap(F.col("CountryRegion")))
    if "StateProvince" in df.columns:
        df = df.withColumn("StateProvince", F.initcap(F.col("StateProvince")))

    df = df.dropDuplicates(["ResellerNaturalKey"])
    df = df.withColumn("ResellerKey", F.row_number().over(Window.orderBy("ResellerNaturalKey")))
    df = df.select("ResellerKey", "ResellerNaturalKey", "Reseller", "BusinessType", "City", "StateProvince", "CountryRegion", "PostalCode")
    write_table(df, SILVER_DIM_RESELLER)
    print(f"  {SILVER_DIM_RESELLER:45s} {spark.table(SILVER_DIM_RESELLER).count():>8,} rows")

    # dim_territory
    df = spark.table(BRONZE_TERRITORY)
    df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date", "notes", "Notes"])

    df = rename_if_exists(df, "REGION", "Region")
    df = rename_if_exists(df, "Sales_Group", "SalesGroup")

    for c in ["Region", "Country", "SalesGroup"]:
        if c in df.columns:
            df = df.withColumn(c, F.initcap(F.trim(F.col(c))))

    df = df.dropDuplicates(["Region", "Country"])
    df = df.withColumn("SalesTerritoryKey", F.row_number().over(Window.orderBy("Region")))
    df = df.select("SalesTerritoryKey", "Region", "Country", "SalesGroup")
    write_table(df, SILVER_DIM_TERRITORY)
    print(f"  {SILVER_DIM_TERRITORY:45s} {spark.table(SILVER_DIM_TERRITORY).count():>8,} rows")

    # dim_date
    df = spark.table(BRONZE_DATE)
    df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date"])

    df = rename_if_exists(df, "Full_Date", "FullDate")
    df = rename_if_exists(df, "Fiscal_Year", "FiscalYear")
    df = rename_if_exists(df, "Fiscal_Quarter", "FiscalQuarter")

    df = df.withColumn("Date", parse_date("Date"))

    if "FullDate" in df.columns:
        df = df.withColumn("FullDate", parse_date("FullDate"))
    if "Month" in df.columns:
        df = df.withColumn("Month", F.trim(F.col("Month")))
    if "FiscalYear" in df.columns:
        df = df.withColumn("FiscalYear", F.concat(F.lit("FY"), F.regexp_extract(F.col("FiscalYear"), r"(\d{4})", 1)))

    fiscal_month = F.when(F.month("Date") >= 7, F.month("Date") - 6).otherwise(F.month("Date") + 6)
    fiscal_quarter_num = F.ceil(fiscal_month / 3).cast(IntegerType())

    if "FiscalQuarter" in df.columns and "FiscalYear" in df.columns:
        df = df.withColumn(
            "FiscalQuarter",
            F.when(
                F.col("FiscalQuarter").isNull() | (F.col("FiscalQuarter") == ""),
                F.concat(F.col("FiscalYear"), F.lit(" Q"), fiscal_quarter_num.cast(StringType()))
            ).otherwise(F.col("FiscalQuarter"))
        )

    df = df.withColumn("DateKey", F.date_format(F.col("Date"), "yyyyMMdd").cast(IntegerType()))
    df = df.withColumn("MonthKey", F.date_format(F.col("Date"), "yyyyMM").cast(IntegerType()))
    df = df.filter(F.col("Date").isNotNull()).dropDuplicates(["DateKey"])

    select_cols = ["DateKey", "Date"]
    if "FiscalYear" in df.columns:
        select_cols.append("FiscalYear")
    if "FiscalQuarter" in df.columns:
        select_cols.append("FiscalQuarter")
    if "Month" in df.columns:
        select_cols.append("Month")
    if "FullDate" in df.columns:
        select_cols.append("FullDate")
    select_cols.append("MonthKey")

    df = df.select(*select_cols)
    write_table(df, SILVER_DIM_DATE)
    print(f"  {SILVER_DIM_DATE:45s} {spark.table(SILVER_DIM_DATE).count():>8,} rows")

    # dim_salesorder
    df = spark.table(BRONZE_SALES_ORDER)
    df = drop_if_exists(df, ["_ingest_time", "_source_file", "_batch_date", "created_date", "Created_Date"])

    df = rename_if_exists(df, "Sales_Order", "SalesOrder")
    df = rename_if_exists(df, "Sales_Order_Line", "SalesOrderLine")

    if "Channel" in df.columns:
        df = df.withColumn("Channel", F.initcap(F.trim(F.col("Channel"))))
    if "SalesOrder" in df.columns:
        df = df.withColumn("SalesOrder", F.trim(F.col("SalesOrder")))
    if "SalesOrderLine" in df.columns:
        df = df.withColumn("SalesOrderLine", F.trim(F.col("SalesOrderLine")))

    df = df.dropDuplicates(["SalesOrderLine"])
    df = (
        df.withColumn("_on", F.regexp_extract(F.col("SalesOrder"), r"SO(\d+)", 1).cast(LongType()))
          .withColumn("_ln", F.regexp_extract(F.col("SalesOrderLine"), r"-\s*(\d+)$", 1).cast(LongType()))
          .withColumn("SalesOrderLineKey", (F.col("_on") * 100 + F.col("_ln")).cast(LongType()))
          .drop("_on", "_ln")
    )
    df = df.select("SalesOrderLineKey", "SalesOrder", "SalesOrderLine", "Channel")
    write_table(df, SILVER_DIM_SALESORDER)
    print(f"  {SILVER_DIM_SALESORDER:45s} {spark.table(SILVER_DIM_SALESORDER).count():>8,} rows")

    # fact_sales
    df_b = spark.table(BRONZE_SALES)
    df_b = drop_if_exists(df_b, ["_ingest_time", "_source_file", "_batch_date", "_batch_id"])

    rename_map = {
        "Customer_ID": "CustomerID",
        "Reseller_ID": "ResellerID",
        "Sales_Order_Line": "SalesOrderLine",
        "Sales_Order": "SalesOrder",
        "Order_Date": "OrderDate",
        "Due_Date": "DueDate",
        "Ship_Date": "ShipDate",
        "Order_Quantity": "OrderQuantity",
        "Unit_Price": "UnitPrice",
        "Unit_Price_Discount_Pct": "UnitPriceDiscountPct",
        "Product_Standard_Cost": "ProductStandardCost",
        "Total_Product_Cost": "TotalProductCost",
        "Extended_Amount": "ExtendedAmount",
        "Sales_Amount": "SalesAmount"
    }

    for old_name, new_name in rename_map.items():
        df_b = rename_if_exists(df_b, old_name, new_name)

    for c in ["CustomerID", "ResellerID", "SKU", "Region", "Country", "SalesOrderLine", "Channel"]:
        if c in df_b.columns:
            df_b = df_b.withColumn(c, F.trim(F.col(c)))

    if "Channel" in df_b.columns:
        df_b = df_b.withColumn("Channel", F.initcap(F.col("Channel")))
    if "Region" in df_b.columns:
        df_b = df_b.withColumn("Region", F.initcap(F.col("Region")))
    if "Country" in df_b.columns:
        df_b = df_b.withColumn("Country", F.initcap(F.col("Country")))

    if "OrderDate" in df_b.columns:
        df_b = df_b.withColumn("OrderDate", parse_date("OrderDate"))
    if "DueDate" in df_b.columns:
        df_b = df_b.withColumn("DueDate", parse_date("DueDate"))
    if "ShipDate" in df_b.columns:
        df_b = df_b.withColumn("ShipDate", parse_date("ShipDate"))

    if "UnitPriceDiscountPct" in df_b.columns:
        df_b = df_b.withColumn(
            "UnitPriceDiscountPct",
            F.when(
                F.col("UnitPriceDiscountPct").cast(StringType()).endswith("%"),
                F.regexp_extract(F.col("UnitPriceDiscountPct"), r"([\d\.]+)%", 1).cast(DoubleType()) / 100
            ).otherwise(F.col("UnitPriceDiscountPct").cast(DoubleType()))
        )

    numeric_casts = {
        "OrderQuantity": IntegerType(),
        "UnitPrice": DoubleType(),
        "ProductStandardCost": DoubleType(),
        "TotalProductCost": DoubleType(),
        "ExtendedAmount": DoubleType(),
        "SalesAmount": DoubleType()
    }

    for c, dtype in numeric_casts.items():
        if c in df_b.columns:
            if c == "OrderQuantity":
                df_b = df_b.withColumn(c, F.round(F.col(c).cast(DoubleType()), 0).cast(dtype))
            else:
                df_b = df_b.withColumn(c, F.col(c).cast(dtype))

    df_b = df_b.withColumn("IsReturn", F.when(F.col("SalesAmount") < 0, 1).otherwise(0))
    df_b = df_b.dropDuplicates(["SalesOrderLine"])

    dc = spark.table(SILVER_DIM_CUSTOMER).select("CustomerKey", "CustomerNaturalKey")
    dp = spark.table(SILVER_DIM_PRODUCT).select("ProductKey", "SKU")
    dr = spark.table(SILVER_DIM_RESELLER).select("ResellerKey", "ResellerNaturalKey")
    dt = spark.table(SILVER_DIM_TERRITORY).select("SalesTerritoryKey", "Region", "Country")
    dd = spark.table(SILVER_DIM_DATE).select("DateKey", "Date")
    ds = spark.table(SILVER_DIM_SALESORDER).select("SalesOrderLineKey", "SalesOrderLine")

    df_fact = (
        df_b
        .join(dc.withColumnRenamed("CustomerNaturalKey", "CustomerID"), on="CustomerID", how="left")
        .join(dp, on="SKU", how="left")
        .join(dr.withColumnRenamed("ResellerNaturalKey", "ResellerID"), on="ResellerID", how="left")
        .join(dt, on=["Region", "Country"], how="left")
        .join(dd.withColumnRenamed("Date", "_dim_date"), F.col("OrderDate") == F.col("_dim_date"), how="left")
        .drop("_dim_date")
        .join(ds, on="SalesOrderLine", how="left")
        .select(
            "SalesOrderLineKey",
            "CustomerKey",
            "ProductKey",
            "ResellerKey",
            "SalesTerritoryKey",
            F.col("DateKey").alias("OrderDateKey"),
            "Channel",
            "OrderQuantity",
            "UnitPrice",
            "UnitPriceDiscountPct",
            "ProductStandardCost",
            "TotalProductCost",
            "ExtendedAmount",
            "SalesAmount",
            "OrderDate",
            "DueDate",
            "ShipDate",
            "IsReturn"
        )
    )

    write_table(df_fact, SILVER_FACT_SALES)
    print(f"  {SILVER_FACT_SALES:45s} {spark.table(SILVER_FACT_SALES).count():>8,} rows")

    max_date = df_fact.agg(F.max("OrderDate")).collect()[0][0]
    if max_date is not None:
        set_watermark("silver_fact_sales", str(max_date), spark.table(SILVER_FACT_SALES).count())

    elapsed = (datetime.now() - start).seconds
    log("silver", rows_out=spark.table(SILVER_FACT_SALES).count())
    print(f"  ✔ Silver done in {elapsed}s")


# COMMAND ----------
# MAGIC %md ## 5. Gold stage

# COMMAND ----------

def run_gold():
    print("\n▓▓▓  STAGE 3/3 — GOLD  ▓▓▓")
    start = datetime.now()

    fact   = spark.table(f"{CATALOG}.{SCHEMA}.silver_fact_sales")
    d_date = spark.table(f"{CATALOG}.{SCHEMA}.silver_dim_date")
    d_prod = spark.table(f"{CATALOG}.{SCHEMA}.silver_dim_product")
    d_cust = spark.table(f"{CATALOG}.{SCHEMA}.silver_dim_customer")
    d_terr = spark.table(f"{CATALOG}.{SCHEMA}.silver_dim_territory")

    def wg(df, name):
        target_table = f"{CATALOG}.{SCHEMA}.gold_{name}"
        write_table(df, target_table)
        n = spark.table(target_table).count()
        print(f"  {target_table:45s} {n:>8,} rows")

    # gold_sales_by_month
    df = (
        fact
        .filter(F.col("IsReturn") == 0)
        .join(
            d_date.select("DateKey", "FiscalYear", "FiscalQuarter", "MonthKey"),
            fact.OrderDateKey == d_date.DateKey,
            "inner"
        )
        .join(
            d_terr.select("SalesTerritoryKey", "Region", "Country", "SalesGroup"),
            "SalesTerritoryKey",
            "left"
        )
        .join(
            d_prod.select("ProductKey", "Category", "Subcategory"),
            "ProductKey",
            "left"
        )
        .groupBy(
            "FiscalYear", "FiscalQuarter", "MonthKey",
            "Region", "Country", "SalesGroup", "Category", "Channel"
        )
        .agg(
            F.count("SalesOrderLineKey").alias("OrderLines"),
            F.sum("OrderQuantity").alias("TotalUnits"),
            F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
            F.round(F.avg("SalesAmount"), 2).alias("AvgOrderValue"),
            F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
            F.countDistinct("CustomerKey").alias("UniqueCustomers")
        )
        .withColumn("GrossProfit", F.round(F.col("TotalRevenue") - F.col("TotalCost"), 2))
        .withColumn(
            "GrossMarginPct",
            F.when(F.col("TotalRevenue") != 0, F.round(F.col("GrossProfit") / F.col("TotalRevenue") * 100, 1))
        )
        .withColumn("_gold_timestamp", F.current_timestamp())
    )
    wg(df, "sales_by_month")

    # gold_product_ranking
    total_rev = fact.filter(F.col("IsReturn") == 0).agg(F.sum("SalesAmount")).collect()[0][0]
    df = (
        fact
        .filter(F.col("IsReturn") == 0)
        .join(
            d_prod.select("ProductKey", "SKU", "Product", "Model", "Category", "Subcategory", "Color"),
            "ProductKey",
            "left"
        )
        .groupBy("ProductKey", "SKU", "Product", "Model", "Category", "Subcategory", "Color")
        .agg(
            F.sum("OrderQuantity").alias("UnitsSold"),
            F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
            F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
            F.round(F.avg("UnitPrice"), 2).alias("AvgSellingPrice"),
            F.count("SalesOrderLineKey").alias("OrderLines")
        )
        .withColumn("GrossProfit", F.round(F.col("TotalRevenue") - F.col("TotalCost"), 2))
        .withColumn(
            "GrossMarginPct",
            F.when(F.col("TotalRevenue") != 0, F.round(F.col("GrossProfit") / F.col("TotalRevenue") * 100, 1))
        )
        .withColumn(
            "RevSharePct",
            F.when(F.lit(total_rev) != 0, F.round(F.col("TotalRevenue") / F.lit(total_rev) * 100, 2))
        )
        .withColumn("RevenueRank", F.rank().over(Window.orderBy(F.desc("TotalRevenue"))))
        .withColumn("_gold_timestamp", F.current_timestamp())
    )
    wg(df, "product_ranking")

    # gold_customer_summary
    df_rfm = (
        fact
        .filter((F.col("IsReturn") == 0) & F.col("CustomerKey").isNotNull())
        .groupBy("CustomerKey")
        .agg(
            F.datediff(F.current_date(), F.max("OrderDate")).alias("RecencyDays"),
            F.count("SalesOrderLineKey").alias("Frequency"),
            F.round(F.sum("SalesAmount"), 2).alias("LifetimeValue"),
            F.round(F.avg("SalesAmount"), 2).alias("AvgOrderValue"),
            F.max("OrderDate").alias("LastOrderDate")
        )
        .withColumn("R_Score", F.when(F.col("RecencyDays") <= 30, 5).when(F.col("RecencyDays") <= 90, 4).when(F.col("RecencyDays") <= 180, 3).when(F.col("RecencyDays") <= 365, 2).otherwise(1))
        .withColumn("F_Score", F.when(F.col("Frequency") >= 20, 5).when(F.col("Frequency") >= 10, 4).when(F.col("Frequency") >= 5, 3).when(F.col("Frequency") >= 2, 2).otherwise(1))
        .withColumn("M_Score", F.when(F.col("LifetimeValue") >= 20000, 5).when(F.col("LifetimeValue") >= 10000, 4).when(F.col("LifetimeValue") >= 3000, 3).when(F.col("LifetimeValue") >= 500, 2).otherwise(1))
        .withColumn("RFM_Score", F.col("R_Score") + F.col("F_Score") + F.col("M_Score"))
        .withColumn(
            "CustomerSegment",
            F.when(F.col("RFM_Score") >= 13, "Champions")
             .when(F.col("RFM_Score") >= 10, "Loyal")
             .when(F.col("RFM_Score") >= 7, "Potential")
             .when(F.col("RFM_Score") >= 5, "At Risk")
             .otherwise("Lost")
        )
    )

    df = (
        d_cust.select("CustomerKey", "CustomerNaturalKey", "CustomerName", "City", "StateProvince", "CountryRegion")
        .join(df_rfm, "CustomerKey", "left")
        .withColumn("Frequency", F.coalesce(F.col("Frequency"), F.lit(0)))
        .withColumn("LifetimeValue", F.coalesce(F.col("LifetimeValue"), F.lit(0.0)))
        .withColumn("CustomerSegment", F.coalesce(F.col("CustomerSegment"), F.lit("Lost")))
        .withColumn("_gold_timestamp", F.current_timestamp())
    )
    wg(df, "customer_summary")

    # gold_channel_compare
    df = (
        fact
        .filter(F.col("IsReturn") == 0)
        .join(
            d_date.select("DateKey", "FiscalYear"),
            fact.OrderDateKey == d_date.DateKey,
            "inner"
        )
        .join(
            d_prod.select("ProductKey", "Category"),
            "ProductKey",
            "left"
        )
        .groupBy("FiscalYear", "Channel", "Category")
        .agg(
            F.count("SalesOrderLineKey").alias("OrderLines"),
            F.sum("OrderQuantity").alias("UnitsSold"),
            F.round(F.sum("SalesAmount"), 2).alias("TotalRevenue"),
            F.round(F.avg("SalesAmount"), 2).alias("AvgOrderValue"),
            F.round(F.sum("TotalProductCost"), 2).alias("TotalCost"),
            F.countDistinct("CustomerKey").alias("UniqueCustomers")
        )
        .withColumn("GrossProfit", F.round(F.col("TotalRevenue") - F.col("TotalCost"), 2))
        .withColumn(
            "GrossMarginPct",
            F.when(F.col("TotalRevenue") != 0, F.round(F.col("GrossProfit") / F.col("TotalRevenue") * 100, 1))
        )
        .withColumn("_gold_timestamp", F.current_timestamp())
    )
    wg(df, "channel_compare")

    elapsed = (datetime.now() - start).seconds
    log("gold")
    print(f"  ✔ Gold done in {elapsed}s")


# COMMAND ----------
# MAGIC %md ## 6. Run the pipeline

# COMMAND ----------

t0 = datetime.now()

print(f"""
╔══════════════════════════════════════════════╗
║  Adventure Works ETL Pipeline                ║
║  Run ID : {RUN_ID:<30}║
║  Mode   : {MODE:<30}║
║  Started: {t0.strftime('%Y-%m-%d %H:%M:%S'):<30}║
╚══════════════════════════════════════════════╝
""")

try:
    run_bronze()
except Exception as e:
    log("bronze", status="FAILED", error=str(e))
    raise

try:
    run_silver()
except Exception as e:
    log("silver", status="FAILED", error=str(e))
    raise

try:
    run_gold()
except Exception as e:
    log("gold", status="FAILED", error=str(e))
    raise

elapsed = (datetime.now() - t0).seconds

print(f"""
╔══════════════════════════════════════════════╗
║  ✅ Pipeline complete ({elapsed}s total)      ║
╚══════════════════════════════════════════════╝
""")


# COMMAND ----------
# MAGIC %md ## 7. Pipeline history

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT
# MAGIC   run_id,
# MAGIC   run_mode,
# MAGIC   stage,
# MAGIC   rows_out,
# MAGIC   status,
# MAGIC   CAST(started_at AS STRING) AS started_at
# MAGIC FROM workspace.medallion.pipeline_run_log
# MAGIC ORDER BY started_at DESC
# MAGIC LIMIT 20

# COMMAND ----------

# MAGIC %sql
# MAGIC SELECT
# MAGIC   table_name,
# MAGIC   CAST(last_order_date AS STRING) AS last_order_date,
# MAGIC   rows_processed,
# MAGIC   run_status,
# MAGIC   CAST(updated_at AS STRING) AS updated_at
# MAGIC FROM workspace.medallion.pipeline_watermarks
# MAGIC ORDER BY updated_at DESC